In [ ]:
# Cell 1: Setup and Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Create Spark session
spark = SparkSession.builder \
    .appName("SentimentTraining") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark ready!")

In [ ]:
# Cell 2: Load data (change path to your cleaned data)
# Option A: Load from HDFS
# df = spark.read.parquet("hdfs:///user/yelp/cleaned/reviews_clean.parquet")

# Option B: Load from local (for testing)
df = spark.read.parquet("data/cleaned/sample_reviews.parquet")

print(f"Loaded {df.count()} reviews")
df.printSchema()

In [ ]:
# Cell 3: Prepare labels (positive=1, negative=0)
# Remove neutral (3 stars) and create label
df_labeled = df.filter(col("stars").isin([1, 2, 4, 5])) \
    .withColumn("label", when(col("stars") >= 4, 1.0).otherwise(0.0))

print(f"Labeled reviews: {df_labeled.count()}")
print(f"Positive: {df_labeled.filter(col('label')==1).count()}")
print(f"Negative: {df_labeled.filter(col('label')==0).count()}")

In [ ]:
# Cell 4: Split train/test
train_data, test_data = df_labeled.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_data.count()}, Test: {test_data.count()}")

In [ ]:
# Cell 5: Build ML pipeline
tokenizer = Tokenizer(inputCol="text_clean", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashingTF = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=5000)
idf = IDF(inputCol="raw_features", outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label",predictionCol="prediction", probabilityCol="probability",maxIter=30, regParam=0.01)

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, lr])
print("Pipeline built")

In [ ]:
# Cell 6: Train model
print("Training model...")
model = pipeline.fit(train_data)
print("Training complete!")

In [ ]:
# Cell 7: Evaluate model
predictions = model.transform(test_data)

# Calculate AUC (Area Under Curve)
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)

# Calculate accuracy
correct = predictions.filter(col("label") == col("prediction")).count()
total = predictions.count()
accuracy = correct / total

print(f"Model Performance:")
print(f"Accuracy: {accuracy:.2%}")
print(f"AUC Score: {auc:.2%}")

In [ ]:
# Cell 8: Function to get probability and rating label
def get_rating_label(prob_pos):
    if prob_pos >= 90:
        return "EXCELLENT"
    elif prob_pos >= 75:
        return "GOOD"
    elif prob_pos >= 50:
        return "NOT BAD"
    elif prob_pos >= 25:
        return "BAD"
    else:
        return "TERRIBLE"

# Test with example reviews
test_reviews = [
    "The food was absolutely amazing! Best meal ever.",
    "Terrible service, rude staff, cold food.",
    "Pretty good experience, would come again.",
    "It was okay, nothing special.",
    "I love this place! Highly recommend!"
]

print("\n" + "="*60)
print("SENTIMENT PREDICTIONS (with probability)")
print("="*60)

for review in test_reviews:
    # Create dataframe
    df_test = spark.createDataFrame([(review,)], ["text_clean"])
    
    # Predict
    result = model.transform(df_test)
    
    # Extract probability
    prob = result.select("probability").collect()[0][0][1] * 100
    
    # Get label
    rating = get_rating_label(prob)
    
    print(f"\nReview: {review}")
    print(f"{prob:.1f}% positive, {100-prob:.1f}% negative")
    print(f"Rating: {rating}")

In [ ]:
# Cell 9: Save model
model_path = "models/sentiment_model"
model.write().overwrite().save(model_path)
print(f"Model saved to {model_path}")

In [ ]:
# Cell 10: Save metrics
import json

metrics = {
    "accuracy": round(accuracy * 100, 1),
    "auc": round(auc * 100, 1)
}

with open("results/metrics/sentiment_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Metrics saved to results/metrics/sentiment_metrics.json")
print(f"Accuracy: {metrics['accuracy']}%")
print(f"AUC: {metrics['auc']}%")

In [ ]:
# Cell 11: Stop Spark
spark.stop()
print("Spark session stopped")